In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/AKI Project"
# DATA_PATH = "/path/to/trinetx_eGFR_gt20_95KPatients"

os.listdir(DATA_PATH)

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
diag_path = os.path.join(DATA_PATH, "diagnosis.csv")
diagnosis = pd.read_csv(diag_path, dtype=str)
diagnosis.head()

In [ ]:
diagnosis.shape
diagnosis['code_system'].value_counts()

In [ ]:
# AKI ICD-10 (N17*)
aki_icd10 = diagnosis[
    (diagnosis['code_system'] == 'ICD-10-CM') &
    (diagnosis['code'].str.startswith('N17', na=False))
]

# AKI ICD-9 (584*)
aki_icd9 = diagnosis[
    (diagnosis['code_system'] == 'ICD-9-CM') &
    (diagnosis['code'].str.startswith('584', na=False))
]

print("AKI ICD-10 count:", aki_icd10.shape[0])
print("AKI ICD-9 count:", aki_icd9.shape[0])

In [ ]:
# Combine ICD-10 and ICD-9 AKI rows
aki_all = pd.concat([aki_icd10, aki_icd9])

print("Total AKI diagnosis rows:", aki_all.shape[0])

In [ ]:
aki_all['date'] = pd.to_datetime(aki_all['date'], format='%Y%m%d', errors='coerce')

aki_all.head()

In [ ]:
aki_index = (
    aki_all
    .sort_values('date')
    .groupby('patient_id')
    .first()
    .reset_index()[['patient_id', 'date']]
)

aki_index.rename(columns={'date': 'aki_index_date'}, inplace=True)

print("Number of unique AKI patients:", aki_index.shape[0])
aki_index.head()

# EXCLUDING ESRD PATIENTS DIAGNOSED BEFORE AKI

Patients diagnosed with ESRD before or on the AKI index date are excluded because:

- ESRD represents the outcome (end-stage kidney failure).
- The study aims to predict **future** progression to ESRD after AKI.
- Including prior ESRD would violate temporal order.
- It would introduce data leakage and artificially inflate model performance.

This ensures a clean and clinically valid AKI progression cohort.

In [ ]:
# ESRD ICD-10 (exact match)
esrd_icd10 = diagnosis[
    (diagnosis['code_system'] == 'ICD-10-CM') &
    (diagnosis['code'] == 'N18.6')
]

# ESRD ICD-9 (exact match)
esrd_icd9 = diagnosis[
    (diagnosis['code_system'] == 'ICD-9-CM') &
    (diagnosis['code'] == '585.6')
]

esrd_all = pd.concat([esrd_icd10, esrd_icd9])

print("Total ESRD diagnosis rows:", esrd_all.shape[0])

In [ ]:
# Ensure AKI index date is datetime
aki_index['aki_index_date'] = pd.to_datetime(aki_index['aki_index_date'], errors='coerce')

# Ensure ESRD date is datetime
esrd_all['date'] = pd.to_datetime(esrd_all['date'], format='%Y%m%d', errors='coerce')

In [ ]:
esrd_in_aki = esrd_all.merge(aki_index, on='patient_id', how='inner')
print("ESRD rows among AKI patients:", esrd_in_aki.shape[0])
print("AKI patients with ANY ESRD at any time:", esrd_in_aki['patient_id'].nunique())

In [ ]:
pre_existing_esrd = esrd_in_aki[esrd_in_aki['date'] <= esrd_in_aki['aki_index_date']]

print("Pre-existing ESRD rows (<= index):", pre_existing_esrd.shape[0])
print("Patients with pre-existing ESRD:", pre_existing_esrd['patient_id'].nunique())

In [ ]:
exclude_patients = set(pre_existing_esrd['patient_id'].unique())

aki_clean = aki_index[~aki_index['patient_id'].isin(exclude_patients)].copy()

print("AKI patients before exclusion:", aki_index.shape[0])
print("AKI patients after ESRD exclusion:", aki_clean.shape[0])

In [ ]:
print("Any excluded patients still in cohort?",
      aki_clean['patient_id'].isin(exclude_patients).any())

# Defining 2 year prediction window

### index date and end date for each patient

In [ ]:
aki_clean['prediction_end_date'] = aki_clean['aki_index_date'] + pd.DateOffset(years=2)

aki_clean.head()

In [ ]:
aki_clean.shape[0]

In [ ]:
# Merge ESRD with clean AKI cohort
esrd_future = esrd_all.merge(aki_clean, on='patient_id', how='inner')

# Keep ESRD that happens AFTER index and WITHIN 2 years
esrd_future = esrd_future[
    (esrd_future['date'] > esrd_future['aki_index_date']) &
    (esrd_future['date'] <= esrd_future['prediction_end_date'])
]

print("Patients who progressed to ESRD within 2 years:",
      esrd_future['patient_id'].nunique())

In [ ]:
# creating a supervised learning target
# creating outcome label: 1 if progressed to ESRD within 2 years, else 0
esrd_patients_2yr = set(esrd_future['patient_id'].unique())

aki_clean['Y_esrd_2yr'] = aki_clean['patient_id'].isin(esrd_patients_2yr).astype(int)

print(aki_clean['Y_esrd_2yr'].value_counts())
print("Positive rate:", aki_clean['Y_esrd_2yr'].mean())

Loading patient.csv to inspect code systems

In [ ]:
proc_path = os.path.join(DATA_PATH, "procedure.csv")

# Use dtype=str so codes like '585.6' or codes with leading zeros don't get mangled
procedure = pd.read_csv(proc_path, dtype=str)

print(procedure.shape)
print(procedure['code_system'].value_counts())

procedure.head()

In [ ]:
# =========================
# DIALYSIS outcome within 2 years
# =========================

# 0) Ensure datetime columns exist
aki_clean = aki_clean.copy()
aki_clean["aki_index_date"] = pd.to_datetime(aki_clean["aki_index_date"], errors="coerce")
if "prediction_end_date" not in aki_clean.columns:
    aki_clean["prediction_end_date"] = aki_clean["aki_index_date"] + pd.DateOffset(years=2)

procedure = procedure.copy()
procedure["date"] = pd.to_datetime(procedure["date"], format="%Y%m%d", errors="coerce")

# 1) Dialysis code definitions (common starters; you can expand later)
CPT_DIALYSIS = {"90935", "90937", "90945", "90947"}      # dialysis procedures
HCPCS_DIALYSIS = {"G0257"}                               # unscheduled dialysis (common)
ICD10PCS_PREFIXES = ("5A1D",)                            # hemodialysis (ICD-10-PCS)
ICD9PROC_DIALYSIS = {"39.95"}                            # hemodialysis (ICD-9 procedure)

# normalize code_system to avoid case issues
procedure["code_system_norm"] = procedure["code_system"].astype(str).str.upper().str.strip()
procedure["code_norm"] = procedure["code"].astype(str).str.strip()

In [ ]:
# 2) Pull dialysis rows from procedure table
dialysis_cpt = procedure[
    (procedure["code_system_norm"].str.contains("CPT", na=False)) &
    (procedure["code_norm"].isin(CPT_DIALYSIS))
]

dialysis_hcpcs = procedure[
    (procedure["code_system_norm"].str.contains("HCPCS", na=False)) &
    (procedure["code_norm"].isin(HCPCS_DIALYSIS))
]

dialysis_icd10pcs = procedure[
    (procedure["code_system_norm"].str.contains("ICD-10", na=False)) &
    (procedure["code_system_norm"].str.contains("PCS", na=False)) &
    (procedure["code_norm"].str.startswith(ICD10PCS_PREFIXES, na=False))
]

dialysis_icd9proc = procedure[
    (procedure["code_system_norm"].str.contains("ICD-9", na=False)) &
    (procedure["code_norm"].isin(ICD9PROC_DIALYSIS))
]

dialysis_all = pd.concat(
    [dialysis_cpt, dialysis_hcpcs, dialysis_icd10pcs, dialysis_icd9proc],
    ignore_index=True
)

print("Total dialysis procedure rows (all patients):", dialysis_all.shape[0])

In [ ]:
# 3) Restrict to AKI cohort and 2-year window
dialysis_in_aki = dialysis_all.merge(
    aki_clean[["patient_id", "aki_index_date", "prediction_end_date"]],
    on="patient_id",
    how="inner"
)

dialysis_future = dialysis_in_aki[
    (dialysis_in_aki["date"] >= dialysis_in_aki["aki_index_date"]) &
    (dialysis_in_aki["date"] <= dialysis_in_aki["prediction_end_date"])
].copy()

dialysis_patients_2yr = set(dialysis_future["patient_id"].unique())
print("AKI patients with ANY dialysis within 2 years (incl acute):", len(dialysis_patients_2yr))

In [ ]:
# 4) Create dialysis label
aki_clean["Y_dialysis_2yr"] = aki_clean["patient_id"].isin(dialysis_patients_2yr).astype(int)
print(aki_clean["Y_dialysis_2yr"].value_counts())
print("Dialysis positive rate:", aki_clean["Y_dialysis_2yr"].mean())

# 5) Combine ESRD OR dialysis label (what a prior student researcher did)
# (Assumes you already created Y_esrd_2yr; recreate esrd_patients_2yr if you need)
esrd_patients_2yr = set(aki_clean.loc[aki_clean["Y_esrd_2yr"] == 1, "patient_id"])
combined_patients_2yr = esrd_patients_2yr | dialysis_patients_2yr

aki_clean["Y_esrd_or_dialysis_2yr"] = aki_clean["patient_id"].isin(combined_patients_2yr).astype(int)
print(aki_clean["Y_esrd_or_dialysis_2yr"].value_counts())
print("Combined positive rate:", aki_clean["Y_esrd_or_dialysis_2yr"].mean())

# Building the BLANKING & OBSERVATION windows

In [ ]:
# Define a 90-day blanking window before AKI; use all earlier history as input
aki_clean["blank_start_date"] = aki_clean["aki_index_date"] - pd.Timedelta(days=90)

In [ ]:
# check difference between index and obs_start
diff_days = (aki_clean["aki_index_date"] - aki_clean["blank_start_date"]).dt.days

print(diff_days.describe())

In [ ]:
aki_clean.head()

In [ ]:
aki_clean.shape

In [ ]:
# Filter diagnosis events to full history before the 90-day blanking window
diagnosis["date"] = pd.to_datetime(diagnosis["date"], format="%Y%m%d", errors="coerce")

diag_obs = diagnosis.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id",
    how="inner"
)

diag_obs = diag_obs[
    diag_obs["date"] < diag_obs["blank_start_date"]
]

In [ ]:
diag_obs.head(20)

In [ ]:
diag_obs["code"].nunique()

# Tabular feature engineering (simpler baseline approach)
- To make sure the cohort is correct
- to check if signals/indicators exist
- gives a benchmark model

Later I will progress to sequences like a prior student researcher did so I could compare the improvement over this simple baseline

In [ ]:
# finding the 300 top most frequent diagnoses codes and arranging them in a table with the codes as columns (300 columns)

top_codes = (
    diag_obs["code"]
    .value_counts()
    .head(300)
    .index
)

In [ ]:
diag_obs_top = diag_obs[diag_obs["code"].isin(top_codes)]

In [ ]:
diag_features = (
    diag_obs_top
    .groupby(["patient_id", "code"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
diag_features.head()

In [ ]:
# merging with the AKI cohort
X = aki_clean[["patient_id"]].merge(
    diag_features,
    on="patient_id",
    how="left"
)

X = X.fillna(0)

In [ ]:
X.head()

In [ ]:
# Remove old target if it exists
if "target" in X.columns:
    X = X.drop(columns=["target"])

In [ ]:
# Add target safely by merging on patient_id
X = X.merge(
    aki_clean[["patient_id", "Y_esrd_or_dialysis_2yr"]],
    on="patient_id",
    how="left"
)

X = X.rename(columns={"Y_esrd_or_dialysis_2yr": "target"})

In [ ]:
X.head()

In [ ]:
print(X.shape)                  # should be (N_COHORT, 300+2)
print(X["target"].isna().sum()) # should be 0
print(X["target"].value_counts())

# ML Modeling

In [ ]:
# separating features and target
# features
X_model = X.drop(columns=["patient_id", "target"])

# target
y_model = X["target"]

print(X_model.shape)
print(y_model.shape)

In [ ]:
y_model.isna().sum()

In [ ]:
# removing nan from target variable
mask = y_model.notna()

X_model = X_model[mask]
y_model = y_model[mask]

print("New shape:", X_model.shape)

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42,
    stratify=y_model
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
y_model.value_counts()

In [ ]:
# starting with LR as it is the standard baseline for classification
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

### above results show the dataset is heavily imbalanced and the accuracy is misleading.
- ROC AUC being 0.5 means the model is randomly guessing.
- F1 Score: 0.003 (extremely low) because recall=0.0 which means the model almost never predicts dialysis patients

# CLASS IMBALANCE FIX
- applying class weights to fix the class imbalance issue

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

- this is a big improvement as recall 0.6 which means the model is finally detecting dialysis patients (model finds 60% of real dialysis patients)
- precision = 0.1 (only 10% of predicted dialysis are correct)
- F1 = 0.17 overall balance
- ROC AUC is still bad as the model cannot separate low-risk vs high-risk patients

# NEXT improvements
- adding procedure features
- adding medication features
- adding CKD stage features
- using better models

# REPLICATING A PRIOR STUDENT RESEARCHER'S PIPELINE

In [ ]:
# Filter procedure events to full history before the 90-day blanking window
import pandas as pd
import os

proc_path = os.path.join(DATA_PATH, "procedure.csv")
procedure = pd.read_csv(proc_path, dtype=str)

procedure["date"] = pd.to_datetime(procedure["date"], format="%Y%m%d", errors="coerce")
aki_clean["blank_start_date"] = pd.to_datetime(aki_clean["blank_start_date"], errors="coerce")

proc_obs = procedure.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id",
    how="inner"
)

proc_obs = proc_obs[
    proc_obs["date"] < proc_obs["blank_start_date"]
]

print(proc_obs.shape)
proc_obs.head()

In [ ]:
# Filter medication events to full history before the 90-day blanking window
med_path = os.path.join(DATA_PATH, "medication_drug.csv")
med = pd.read_csv(
    med_path,
    usecols=["patient_id", "code", "code_system", "start_date"],
    dtype=str,
    on_bad_lines='skip'
    # removed engine='python' — default C engine uses much less RAM
)

med["start_date"] = pd.to_datetime(med["start_date"], format="%Y%m%d", errors="coerce")
aki_clean["blank_start_date"] = pd.to_datetime(aki_clean["blank_start_date"], errors="coerce")

med_obs = med.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id",
    how="inner"
)

med_obs = med_obs[
    med_obs["start_date"] < med_obs["blank_start_date"]
]

print(med_obs.shape)
med_obs.head()

In [ ]:
# Filter diagnosis events to full history before the 90-day blanking window
diag_obs = diagnosis.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id",
    how="inner"
)

diag_obs = diag_obs[
    diag_obs["date"] < diag_obs["blank_start_date"]
]

diag_obs.head()

In [ ]:
print(diag_obs.shape)
print(proc_obs.shape)
print(med_obs.shape)

Meaning:

[figure withheld] diagnosis events

[figure withheld] procedure events

[figure withheld] medication events

All from full patient history before the 90-day blanking window.

Total timeline size: [figure withheld] events

In [ ]:
# converting each table into same event format

# diagnosis
diagnosis_events = diag_obs[["patient_id","date","code"]].copy()
diagnosis_events["event_type"] = "diagnosis"
diagnosis_events = diagnosis_events.dropna(subset=["date"])

# procedure
procedure_events = proc_obs[["patient_id","date","code"]].copy()
procedure_events["event_type"] = "procedure"
procedure_events = procedure_events.dropna(subset=["date"])

# medication
medication_events = med_obs[["patient_id","start_date","code"]].copy()
medication_events = medication_events.rename(columns={"start_date":"date"})
medication_events["event_type"] = "medication"
medication_events = medication_events.dropna(subset=["date"])

# merge all
all_events = pd.concat(
    [diagnosis_events, procedure_events, medication_events],
    ignore_index=True
)

print(all_events.shape)

In [ ]:
# merging all events

all_events = pd.concat(
    [diagnosis_events, procedure_events, medication_events],
    ignore_index=True
)

print(all_events.shape)
all_events.head()

In [ ]:
# sorting events chronologically per patient
all_events = all_events.sort_values(["patient_id", "date"])

all_events.head(20)

In [ ]:
# removing sparse patients (very short sequences)
# computing sequence length
seq_lengths = all_events.groupby("patient_id").size()

seq_lengths.head()

In [ ]:
seq_lengths.shape

In [ ]:
# keeping patients with more than 3 events
valid_patients = seq_lengths[seq_lengths >= 3].index

In [ ]:
valid_patients.shape

In [ ]:
# filtering the timeline
all_events_filtered = all_events[
    all_events["patient_id"].isin(valid_patients)
]

print(all_events_filtered.shape)

In [ ]:
# Sorting filtered events by patient and time before sequence creation

all_events_filtered = all_events_filtered.sort_values(
    by=["patient_id", "date"]
).reset_index(drop=True)

In [ ]:
# Free upstream dataframes that aren't needed beyond this point.
import gc

vars_to_clear = [
    # full source CSVs (huge)
    'diagnosis', 'procedure', 'med',
    # filtered observation windows (medium)
    'diag_obs', 'proc_obs', 'med_obs',
    # per-modality event dfs (medium)
    'diagnosis_events', 'procedure_events', 'medication_events',
    'all_events',
    # LR baseline artifacts (not needed for RETAIN)
    'top_codes', 'diag_obs_top', 'diag_features',
    'X', 'X_model', 'y_model', 'X_train', 'X_test', 'y_train', 'y_test',
    # cohort intermediates that aren't needed downstream
    'aki_index', 'aki_all', 'aki_icd10', 'aki_icd9',
    'esrd_all', 'esrd_icd10', 'esrd_icd9', 'esrd_in_aki', 'esrd_future',
    'pre_existing_esrd',
    'dialysis_all', 'dialysis_in_aki', 'dialysis_future',
    'dialysis_cpt', 'dialysis_hcpcs', 'dialysis_icd10pcs', 'dialysis_icd9proc',
]

for v in vars_to_clear:
    if v in globals():
        del globals()[v]

gc.collect()
print("Memory freed. About to load lab_result.csv.")
!free -h

## Phase 2: Lab loading with curated codes + clinical binning

Loads kidney labs using a curated list of LOINC codes (not description regex), and converts each numeric result into a clinical severity label so the model can learn from lab values, not just their presence.

### What this cell does
1. Defines curated LOINC code sets for creatinine, eGFR, and BUN
2. Reads `lab_result.csv` in chunks, filtering to those codes (RAM-safe)
3. Drops rows with missing values, wrong units for the test, or implausible numbers
4. Converts µmol/L creatinine → mg/dL for unit consistency
5. **Bins each value into a clinical severity category**:

| Test | Severity ranking |
|---|---|
| **Creatinine** (mg/dL) | normal (<1.2) → mild (1.2–2.0) → moderate (2.0–4.0) → severe (>4.0) |
| **eGFR** (mL/min/1.73m²) | G1 (≥90, healthy) → G2 → G3a → G3b → G4 → G5 (<15, kidney failure) — KDIGO staging |
| **BUN** (mg/dL) | normal (<20) → elevated (20–50) → high (>50) |

6. Restricts to AKI cohort + applies 90-day blanking window
7. Appends labs to event timeline using binned tokens like `lab_creatinine_severe`, `lab_egfr_g4`, etc.

In [ ]:
# ─── Phase 2: curated LOINC codes + clinical binning ───
import gc

# Curated LOINC code sets — the canonical codes for each kidney lab test
creatinine_codes = {
    "2160-0",   # Creatinine [Mass/volume] in Serum or Plasma
    "38483-4",  # Creatinine [Mass/volume] in Blood
    "14682-9",  # Creatinine [Moles/volume] in Serum or Plasma
    "21232-4",  # Creatinine [Mass/volume] in Arterial blood
    "35203-9",  # Creatinine [Mass/volume] in Capillary blood
    "35591-7",  # Creatinine [Mass/volume] in Venous blood
}

egfr_codes = {
    "33914-3",  # GFR predicted by Creatinine-based formula (MDRD)
    "48642-3",  # MDRD, non-Black
    "48643-1",  # MDRD, Black
    "62238-1",  # CKD-EPI
    "88293-6",  # CKD-EPI 2021 (race-free)
    "88294-4",  # CKD-EPI 2021 refit
    "50044-7",  # eGFR/1.73 sq M predicted in Black
    "50210-4",  # eGFR/1.73 sq M predicted
}

bun_codes = {
    "3094-0",   # Urea nitrogen [Mass/volume] in Serum or Plasma
    "6299-2",   # Urea nitrogen [Mass/volume] in Blood
    "11065-0",  # Urea nitrogen [Mass/volume] in Arterial blood
}

all_kidney_codes = creatinine_codes | egfr_codes | bun_codes

# ─── Read lab_result.csv in chunks, filter to curated codes (now including values + units) ───
print("Reading lab_result.csv in chunks...")
chunks = []
for i, chunk in enumerate(pd.read_csv(
    os.path.join(DATA_PATH, "lab_result.csv"),
    usecols=["patient_id", "code", "code_system", "date",
             "lab_result_num_val", "units_of_measure"],
    on_bad_lines='skip', engine='python',
    chunksize=500_000,
)):
    chunk["code"] = chunk["code"].astype(str).str.strip()
    filtered = chunk[
        (chunk["code_system"] == "LOINC") &
        (chunk["code"].isin(all_kidney_codes))
    ]
    if len(filtered) > 0:
        chunks.append(filtered)
    if (i + 1) % 20 == 0:
        print(f"  processed chunk {i+1} ({(i+1)*500_000:,} rows scanned)")

kidney_labs = pd.concat(chunks, ignore_index=True).copy()
del chunks
gc.collect()
print(f"Kidney lab rows loaded: {len(kidney_labs):,}")

# ─── Convert numeric values, drop NaN ───
kidney_labs["lab_result_num_val"] = pd.to_numeric(
    kidney_labs["lab_result_num_val"], errors="coerce"
)
kidney_labs = kidney_labs.dropna(subset=["lab_result_num_val"]).copy()
print(f"After dropping NaN values: {len(kidney_labs):,}")

# ─── Classify by code (unambiguous, no description regex needed) ───
def classify(c):
    if c in creatinine_codes: return "creatinine"
    if c in egfr_codes:       return "egfr"
    if c in bun_codes:        return "bun"
    return None
kidney_labs["test_type"] = kidney_labs["code"].apply(classify)

# ─── Validate units (drop rows where units don't match test) ───
kidney_labs["units_of_measure"] = kidney_labs["units_of_measure"].astype(str).str.strip()

valid_units = (
    ((kidney_labs["test_type"] == "creatinine") & kidney_labs["units_of_measure"].isin(["mg/dL", "umol/L"])) |
    ((kidney_labs["test_type"] == "egfr")       & (kidney_labs["units_of_measure"] == "mL/min/{1.73_m2}")) |
    ((kidney_labs["test_type"] == "bun")        & (kidney_labs["units_of_measure"] == "mg/dL"))
)
kidney_labs = kidney_labs[valid_units].copy()
print(f"After units validation: {len(kidney_labs):,}")

# Convert umol/L creatinine to mg/dL
umol_mask = (kidney_labs["test_type"] == "creatinine") & \
            (kidney_labs["units_of_measure"] == "umol/L")
kidney_labs.loc[umol_mask, "lab_result_num_val"] /= 88.4

# ─── Drop implausible values (data errors) ───
def is_plausible(row):
    v, t = row["lab_result_num_val"], row["test_type"]
    if t == "creatinine": return 0.1 <= v <= 30
    if t == "egfr":       return 0   <= v <= 200
    if t == "bun":        return 1   <= v <= 200
    return False
kidney_labs = kidney_labs[kidney_labs.apply(is_plausible, axis=1)].copy()
print(f"After dropping implausible values: {len(kidney_labs):,}")

# ─── Bin each value into a clinical severity category ───
def bin_creatinine(v):
    if v < 1.2: return "normal"
    if v < 2.0: return "mild"
    if v < 4.0: return "moderate"
    return "severe"

def bin_egfr(v):
    if v >= 90: return "g1"
    if v >= 60: return "g2"
    if v >= 45: return "g3a"
    if v >= 30: return "g3b"
    if v >= 15: return "g4"
    return "g5"

def bin_bun(v):
    if v < 20: return "normal"
    if v < 50: return "elevated"
    return "high"

def make_bin(row):
    t, v = row["test_type"], row["lab_result_num_val"]
    if t == "creatinine": return f"creatinine_{bin_creatinine(v)}"
    if t == "egfr":       return f"egfr_{bin_egfr(v)}"
    if t == "bun":        return f"bun_{bin_bun(v)}"

kidney_labs["binned_code"] = kidney_labs.apply(make_bin, axis=1)
print("\nDistribution of binned lab tokens:")
print(kidney_labs["binned_code"].value_counts())

# ─── Convert dates, filter to AKI cohort, apply blanking window ───
kidney_labs["date"] = pd.to_datetime(
    kidney_labs["date"].astype(str), format="%Y%m%d", errors="coerce"
)

lab_filtered = kidney_labs[
    kidney_labs["patient_id"].isin(aki_clean["patient_id"])
].copy()

lab_filtered = lab_filtered.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id", how="inner"
)

lab_filtered = lab_filtered[
    lab_filtered["date"] < lab_filtered["blank_start_date"]
].copy()

print(f"\nKidney lab events after blanking filter: {lab_filtered.shape}")

# ─── Append labs to event timeline as BINNED tokens ───
lab_events = lab_filtered[["patient_id", "date", "binned_code"]].rename(
    columns={"binned_code": "code"}
).copy()
lab_events["event_type"] = "lab"
lab_events = lab_events.dropna(subset=["date"])

print("Lab events to append:", lab_events.shape)
print("Unique lab tokens:", lab_events["code"].nunique())

all_events_filtered = pd.concat(
    [all_events_filtered, lab_events], ignore_index=True
)
all_events_filtered = all_events_filtered.sort_values(
    by=["patient_id", "date"]
).reset_index(drop=True)

print(f"\nTotal events after adding labs: {all_events_filtered.shape}")
print(all_events_filtered["event_type"].value_counts())

del kidney_labs, lab_filtered, lab_events
gc.collect()

In [ ]:
# converting the codes into sequence tokens for the transformers
# make each event unique by combining type + code
all_events_filtered["event_token"] = (
    all_events_filtered["event_type"].astype(str) + "_" +
    all_events_filtered["code"].astype(str)
)

all_events_filtered.head()

In [ ]:
# grouping the events per patient (UPDATED for transformer tokens)
patient_sequences = all_events_filtered.groupby("patient_id")["event_token"].apply(list)

print(patient_sequences.shape)
patient_sequences.head()

In [ ]:
# build vocabulary and numeric sequences
# unique event tokens
unique_tokens = all_events_filtered["event_token"].unique()

# token -> integer mapping
token2id = {token: i + 1 for i, token in enumerate(unique_tokens)}   # start from 1
id2token = {i: token for token, i in token2id.items()}

print("Vocabulary size:", len(token2id))

In [ ]:
# converting each patient timeline into integer sequences
patient_sequences = (
    all_events_filtered
    .groupby("patient_id")["event_token"]
    .apply(list)
)

patient_sequences_ids = patient_sequences.apply(
    lambda seq: [token2id[token] for token in seq]
)

print(patient_sequences_ids.shape)
patient_sequences_ids.head()

In [ ]:
# attaching the label Y to each patient sequence
# create patient-level label table
labels = aki_clean[["patient_id", "Y_esrd_or_dialysis_2yr"]].drop_duplicates()

# turn sequence series into dataframe
seq_df = patient_sequences_ids.reset_index()
seq_df.columns = ["patient_id", "sequence"]

# merge sequences with labels
seq_df = seq_df.merge(labels, on="patient_id", how="inner")

print(seq_df.shape)
seq_df.head()

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    seq_df,
    test_size=0.2,
    random_state=42,
    stratify=seq_df["Y_esrd_or_dialysis_2yr"]
)

print(train_df.shape, test_df.shape)

In [ ]:
# finding the sequence lenghts before padding
train_df["seq_len"] = train_df["sequence"].apply(len)
test_df["seq_len"] = test_df["sequence"].apply(len)

print(train_df["seq_len"].describe())
print(test_df["seq_len"].describe())

In [ ]:
import numpy as np

MAX_SEQ_LEN = 400

def fast_pad_sequences(sequences, maxlen, padding='post', truncating='post'):
    out = np.zeros((len(sequences), maxlen), dtype=np.int32)
    for i, seq in enumerate(sequences):
        seq = list(seq)
        if len(seq) > maxlen:
            seq = seq[:maxlen] if truncating == 'post' else seq[-maxlen:]
        if len(seq) > 0:
            if padding == 'post':
                out[i, :len(seq)] = seq
            else:
                out[i, -len(seq):] = seq
    return out

X_train = fast_pad_sequences(train_df["sequence"], maxlen=MAX_SEQ_LEN)
X_test  = fast_pad_sequences(test_df["sequence"],  maxlen=MAX_SEQ_LEN)

y_train = train_df["Y_esrd_or_dialysis_2yr"].values
y_test  = test_df["Y_esrd_or_dialysis_2yr"].values

print(X_train.shape)
print(X_test.shape)

# Building RETAIN.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

VOCAB_SIZE   = len(token2id) + 1   # +1 for pad token = 0
MAX_SEQ_LEN  = 400
EMBED_DIM    = 128
ALPHA_HIDDEN = 128
BETA_HIDDEN  = 128
DROPOUT      = 0.1

def build_retain_model(maxlen, vocab_size,
                       embed_dim=128, alpha_hidden=128, beta_hidden=128, rate=0.1):
    """
    RETAIN (Choi et al. 2016) — two-level attention over patient event sequences.
      α : scalar attention over events (which events matter)
      β : vector attention over embedding dims (which features within an event matter)
    Reads sequences in REVERSE so the most recent event seeds the GRU hidden state.
    """
    inputs = layers.Input(shape=(maxlen,), dtype="int32", name="event_ids")

    # mask: 1 for real tokens, 0 for padding (wrap tf ops in Lambda for Keras 3)
    mask_exp = layers.Lambda(
        lambda x: tf.expand_dims(tf.cast(tf.not_equal(x, 0), tf.float32), -1),
        name="mask",
    )(inputs)                                                          # (B, T, 1)

    # event embeddings, with padded positions zeroed out
    v = layers.Embedding(vocab_size, embed_dim, name="event_embedding")(inputs)
    v = layers.Multiply(name="zero_pad_embeds")([v, mask_exp])
    v = layers.Dropout(rate)(v)

    # reverse sequence so most recent event comes first into the GRUs
    v_rev = layers.Lambda(lambda t: tf.reverse(t, axis=[1]))(v)

    # ── α : event-level scalar attention ───────────────────────────────
    g = layers.GRU(alpha_hidden, return_sequences=True, name="alpha_gru")(v_rev)
    alpha_logits = layers.Dense(1, name="alpha_logits")(g)             # (B, T, 1)
    # un-reverse so alpha aligns with original event order
    alpha_logits = layers.Lambda(lambda t: tf.reverse(t, axis=[1]))(alpha_logits)
    # mask padded positions BEFORE softmax (large negative => ~0 weight)
    neg_inf_mask = layers.Lambda(
        lambda m: (1.0 - m) * -1e9, name="neg_inf_mask"
    )(mask_exp)
    alpha_logits = layers.Add()([alpha_logits, neg_inf_mask])
    alpha = layers.Softmax(axis=1, name="alpha")(alpha_logits)         # (B, T, 1)

    # ── β : feature-level vector attention ────────────────────────────
    h = layers.GRU(beta_hidden, return_sequences=True, name="beta_gru")(v_rev)
    beta = layers.Dense(embed_dim, activation="tanh", name="beta")(h)
    beta = layers.Lambda(lambda t: tf.reverse(t, axis=[1]))(beta)      # (B, T, E)

    # context vector: c = Σ_t α_t · (β_t ⊙ v_t)
    weighted = layers.Multiply()([beta, v])
    weighted = layers.Multiply()([alpha, weighted])
    weighted = layers.Multiply()([weighted, mask_exp])                  # belt + suspenders
    context  = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name="context")(weighted)
    context  = layers.Dropout(rate)(context)

    outputs = layers.Dense(1, activation="sigmoid", name="output")(context)
    return Model(inputs=inputs, outputs=outputs, name="RETAIN")

model = build_retain_model(
    maxlen=MAX_SEQ_LEN,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    alpha_hidden=ALPHA_HIDDEN,
    beta_hidden=BETA_HIDDEN,
    rate=DROPOUT,
)
model.summary()

In [ ]:
import tensorflow as tf

def binary_focal_loss(alpha=0.75, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

        loss_val = -alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt)
        return tf.reduce_mean(loss_val)

    return loss

In [ ]:
# compiling focal loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=binary_focal_loss(alpha=0.5, gamma=2.0),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=3,
    mode="max",
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# evaluate test set
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# adding new metrics:

from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("Sensitivity:", sensitivity)
print("Specificity:", specificity)

In [ ]:
# new metrics (to match a prior student researcher's)
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, y_prob)
print("PR-AUC:", pr_auc)

In [ ]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(y_test, y_prob)
print("Brier score:", brier)

In [ ]:
# Tune classification threshold by maximizing F1 score on predicted probabilities.
# This helps improve detection of minority (positive) cases in an imbalanced dataset.

from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.05, 0.96, 0.01)

best_thresh = 0.36
best_f1 = 0

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best threshold:", best_thresh)
print("Best F1:", best_f1)

In [ ]:
# Apply the F1-optimized threshold to generate final class predictions
best_thresh = 0.36
y_pred = (y_prob >= best_thresh).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))   # use probabilities, not y_pred
print("F1:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Applying alpha tuning

In [ ]:
# hyperparameter grid search with automatic best-config tracking

alphas = [0.25, 0.5]
dropouts = [0.1, 0.2]

results = []
best_result = None

for alpha in alphas:
    for dropout in dropouts:
        print(f"\nAlpha={alpha}, Dropout={dropout}")

        model = build_retain_model(
            maxlen=MAX_SEQ_LEN,
            vocab_size=VOCAB_SIZE,
            embed_dim=128,
            alpha_hidden=128,
            beta_hidden=128,
            rate=dropout
        )

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss=binary_focal_loss(alpha=alpha, gamma=2.0),
            metrics=[
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.AUC(name="auc")
            ]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            patience=3,
            mode="max",
            restore_best_weights=True
        )

        model.fit(
            X_train,
            y_train,
            validation_split=0.2,
            epochs=7,
            batch_size=32,
            callbacks=[early_stop],
            verbose=0
        )

        y_prob = model.predict(X_test).ravel()

        thresholds = np.arange(0.05, 0.96, 0.01)
        best_f1 = 0
        best_thresh = 0.5

        for t in thresholds:
            y_pred = (y_prob >= t).astype(int)
            f1 = f1_score(y_test, y_pred)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t

        auc = roc_auc_score(y_test, y_prob)

        current_result = {
            "alpha": alpha,
            "dropout": dropout,
            "f1": best_f1,
            "auc": auc,
            "threshold": best_thresh
        }

        results.append(current_result)

        if best_result is None or current_result["f1"] > best_result["f1"]:
            best_result = current_result

        print("Best F1:", best_f1, "| Threshold:", best_thresh, "| AUC:", auc)

print("\nAll results:")
for r in results:
    print(r)

print("\nBest config selected automatically:")
print(best_result)

# save best values for later cells
BEST_ALPHA = best_result["alpha"]
BEST_DROPOUT = best_result["dropout"]
BEST_THRESHOLD = best_result["threshold"]

In [ ]:
# Final RETAIN evaluation with ROC-AUC, PR-AUC, Brier score, sensitivity, and specificity

from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, average_precision_score, brier_score_loss
)

print("Using best config:")
print("BEST_ALPHA =", BEST_ALPHA)
print("BEST_DROPOUT =", BEST_DROPOUT)
print("BEST_THRESHOLD =", BEST_THRESHOLD)

model = build_retain_model(
    maxlen=MAX_SEQ_LEN,
    vocab_size=VOCAB_SIZE,
    embed_dim=128,
    alpha_hidden=128,
    beta_hidden=128,
    rate=BEST_DROPOUT
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=binary_focal_loss(alpha=BEST_ALPHA, gamma=2.0),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=3,
    mode="max",
    restore_best_weights=True
)

model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=7,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= BEST_THRESHOLD).astype(int)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
pr_auc = average_precision_score(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", pr_auc)
print("Brier Score:", brier)
print("F1:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Sensitivity:", sensitivity)
print("Specificity:", specificity)
print("Confusion matrix:\n", cm)

In [ ]:
# saving results before continuing the project in another file
# Main event-level table used to add new modalities later
print(all_events_filtered.shape)
print(all_events_filtered.columns)
print(all_events_filtered["event_type"].value_counts())
all_events_filtered.head(10)

In [ ]:
# Save patient IDs for consistent future experiments
train_ids = train_df["patient_id"].tolist()
test_ids = test_df["patient_id"].tolist()

print("Train patients:", len(train_ids))
print("Test patients:", len(test_ids))

In [ ]:
# Main table for future modality expansion
print("all_events_filtered shape:", all_events_filtered.shape)
print("Columns:", all_events_filtered.columns.tolist())
print(all_events_filtered["event_type"].value_counts())
all_events_filtered.head(10)